Parameters

In [1]:
m = 10
mean_service = 8.0
mean_interarrival = 1.0

A = mean_service / mean_interarrival

In [4]:
import heapq
import random
import math

In [5]:
def loss_system_simulation(
        m,
        arrival_generator,
        service_generator,
        n_customers=10000):

    clock = 0.0

    busy = 0

    blocked = 0

    arrivals_seen = 0

    event_list = []

    first_arrival = arrival_generator()

    heapq.heappush(
        event_list,
        (first_arrival, "arrival")
    )

    while arrivals_seen < n_customers:

        clock, event = heapq.heappop(event_list)

        if event == "arrival":

            arrivals_seen += 1

            next_arrival = clock + arrival_generator()

            heapq.heappush(
                event_list,
                (next_arrival, "arrival")
            )

            if busy == m:

                blocked += 1

            else:

                busy += 1

                departure_time = (
                    clock +
                    service_generator()
                )

                heapq.heappush(
                    event_list,
                    (departure_time, "departure")
                )

        else:

            busy -= 1

    return blocked / arrivals_seen

In [6]:
import numpy as np
from scipy.stats import t

In [7]:
def confidence_interval(values,
                        alpha=0.05):

    n = len(values)

    mean = np.mean(values)

    s = np.std(values, ddof=1)

    tcrit = t.ppf(
        1-alpha/2,
        n-1
    )

    halfwidth = (
        tcrit*s/math.sqrt(n)
    )

    return (
        mean,
        mean-halfwidth,
        mean+halfwidth
    )

Part 1 - Poisson arrivals

In [8]:
def poisson_arrival():

    return random.expovariate(1.0)

In [9]:
def exponential_service():

    return random.expovariate(
        1/8
    )

In [10]:
results = []

for _ in range(10):

    p = loss_system_simulation(
        m=10,
        arrival_generator=poisson_arrival,
        service_generator=exponential_service,
        n_customers=10000
    )

    results.append(p)

In [17]:
mean, lower, upper = confidence_interval(
    results
)
print(mean, lower, upper)

0.12086000000000001 0.11670132075573486 0.12501867924426516


In [13]:
def erlang_b(A,m):

    numerator = (
        A**m /
        math.factorial(m)
    )

    denominator = 0

    for i in range(m+1):
        denominator += (
            A**i /
            math.factorial(i)
        )

    return numerator / denominator

In [14]:
exact = erlang_b(8,10)

print(exact)

0.12166106425295149


Part 2.a - renewal arrivals (k = 2), mean must remain 1

$mean = k/\lambda \rightarrow 1 = 2/\lambda \rightarrow \lambda = 2$

In [18]:
def erlang_arrival():

    return (
        random.expovariate(2)
        +
        random.expovariate(2)
    )

In [19]:
results_erlang = []

for _ in range(10):

    p = loss_system_simulation(
        m=10,
        arrival_generator=erlang_arrival,
        service_generator=exponential_service
    )

    results_erlang.append(p)

In [20]:
results_erlang

[0.1034, 0.0871, 0.0918, 0.105, 0.0887, 0.0962, 0.0907, 0.0823, 0.0968, 0.0936]

Part 2.b - Hyperexponential arrivals

In [21]:
p1 = 0.8
lambda1 = 0.8333

p2 = 0.2
lambda2 = 5.0

In [22]:
def hyper_arrival():

    u = random.random()

    if u < 0.8:
        return random.expovariate(
            0.8333
        )

    else:
        return random.expovariate(
            5.0
        )

In [23]:
results_hyper = []

for _ in range(10):

    p = loss_system_simulation(
        m=10,
        arrival_generator=hyper_arrival,
        service_generator=exponential_service
    )

    results_hyper.append(p)

In [24]:
results_hyper

[0.1328, 0.1282, 0.1332, 0.149, 0.1446, 0.1472, 0.1341, 0.1347, 0.1456, 0.1351]

Part 3a - Poisson arrivals revisited

In [25]:
def constant_service():

    return 8.0

In [26]:
results_constant = []

for _ in range(10):

    p = loss_system_simulation(
        m=10,
        arrival_generator=poisson_arrival,
        service_generator=constant_service
    )

    results_constant.append(p)

In [27]:
results_constant

[0.128, 0.1241, 0.1174, 0.1233, 0.1235, 0.1233, 0.1177, 0.1214, 0.1229, 0.1237]

Part 3.b - Pareto service times

In [28]:
k1 = 1.05
k2 = 2.05

$E[X] = \frac{\beta k}{k-1}$

mean == 8, therefore

$\beta = 8\frac{k-1}{k}$

In [29]:
beta_105 = (8*(k1-1)/k1)

In [31]:
def pareto_service_105():

    u = random.random()

    return (
        beta_105 *
        u**(-1/1.05)
    )

In [39]:
pareto_service_105()

0.5139503597255329

In [40]:
beta_205 = (8*(k2-1)/k2)

In [42]:
def pareto_service_205():

    u = random.random()

    return (
        beta_205 *
        u**(-1/2.05)
    )

In [43]:
results_p105 = []
results_p205 = []

In [44]:
for _ in range(10):
    results_p105.append(pareto_service_105())
    results_p205.append(pareto_service_205())

In [46]:
results_p105, results_p205

([0.640000363027515,
  0.428404071629932,
  0.5557563424767179,
  7.089657592820241,
  1.3682803008012632,
  0.5288001255852529,
  0.3990092690347711,
  0.8012100716125071,
  0.5905231321443923,
  0.7375393962755025],
 [6.562681117736792,
  12.675334303580723,
  8.972208606174956,
  14.143274362902613,
  5.314982126499455,
  4.726819365728515,
  11.010927477497475,
  6.369606006193116,
  7.390872067257087,
  5.0191914420455666])

Part 3.c

Uniform service

$mean = 8 \rightarrow U(4, 12)$

In [47]:
def uniform_service():

    return random.uniform(
        4,
        12
    )

Erlang service

$k = 4 \rightarrow \lambda=0.5$

In [48]:
def erlang_service():

    return sum(
        random.expovariate(0.5)
        for _ in range(4)
    )